# Introduction to Agentic Workflows

An **agentic workflow** combines two ideas:

1. **Workflow** — explicit control flow you define (steps, branches, checkpoints, termination).
2. **Agentic steps** — individual steps where an LLM (or agent) reasons, uses tools, or produces artifacts within guardrails.

Unlike a fully **autonomous agent** that decides every next move, an agentic workflow keeps **orchestration in your code** while delegating **cognitive work** to specialized agents at specific nodes. Unlike a **pure workflow** with only deterministic functions, agentic workflows embed **non-deterministic intelligence** where it adds value — drafting, reviewing, classifying — without surrendering auditability.

This notebook defines agentic workflows, contrasts them with single-agent loops and rigid pipelines, introduces the core building blocks, and implements a working **ReleaseFlow** orchestrator that coordinates researcher, writer, and critic agents to produce a v2.4.0 release announcement. Everything is self-contained plain Python — no frameworks required.

In [ ]:
"""
ReleaseFlow — engineering release announcement orchestration environment.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum, auto
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


CHANGELOG: list[dict[str, str]] = [
    {"version": "2.4.0", "item": "Added bulk export endpoint for reports"},
    {"version": "2.4.0", "item": "Fixed session timeout on mobile browsers"},
    {"version": "2.4.0", "item": "Deprecated legacy v1 webhook format"},
]

RUNBOOKS: list[dict[str, str]] = [
    {
        "id": "deploy-200",
        "title": "Production deploy checklist",
        "text": "Run integration tests, apply DB migrations, deploy to canary 10%, monitor error rate 15 min, then full rollout.",
    },
    {
        "id": "comm-201",
        "title": "Customer communication standards",
        "text": "Lead with user benefit, list breaking changes separately, include rollback window and support channel #releases.",
    },
]

STYLE_GUIDE = {
    "tone": "professional, concise",
    "must_include": ["version number", "breaking changes section", "support channel"],
    "max_words": 200,
}

PUBLISHED_ANNOUNCEMENTS: list[dict[str, Any]] = []

print(f"Changelog entries: {len(CHANGELOG)} | Runbooks: {len(RUNBOOKS)}")

---

## 1. What Is an Agentic Workflow?

Formally, an agentic workflow can be described as:

```
AgenticWorkflow = {
  steps[],           # ordered or graph-connected nodes
  agents[],          # specialists invoked at specific steps
  shared_state,      # artifacts passed between steps
  orchestrator,      # code that decides which step runs next
  termination_rules  # when to stop (success, max retries, budget)
}
```

| Concept | Meaning |
|---------|--------|
| **Step** | A unit of work — deterministic function or agent invocation |
| **Agentic step** | Step where an LLM/agent produces output (draft, review, classify) |
| **Orchestrator** | Your code — not the model — owns the graph edges |
| **Shared state** | Research notes, drafts, approvals visible across steps |
| **Hand-off** | One step writes to state; the next step reads it |

**ReleaseFlow example:**

```
research (agent) ──► write (agent) ──► review (agent) ──► [reject?] ──► write again
                                              │
                                              └── approve ──► human_gate ──► publish
```

The **path** (research before write, review before publish) is fixed in code. The **content** at each agent step is generated dynamically.

---

## 2. Three Patterns Compared

| | **Pure workflow** | **Agentic workflow** | **Autonomous agent** |
|--|-------------------|----------------------|----------------------|
| **Control owner** | Engineer (100%) | Engineer orchestrates; agents work inside nodes | Model (+ guardrails) |
| **Topology** | Linear DAG | DAG with agent nodes + optional loops | Cyclic tool loop |
| **LLM usage** | Optional, at fixed nodes | At designated agent steps | Every iteration |
| **Determinism** | High — same graph every run | Medium — graph fixed, text varies | Low — path varies |
| **Auditability** | Excellent | Good — log step + agent I/O | Harder — log every model decision |
| **Best for** | ETL, cron, compliance gates | Content pipelines, review loops | Open-ended investigation |

**Agentic workflows** sit in the sweet spot for production: you keep **compliance checkpoints** and **predictable structure**, while agents handle **language-heavy** work.

---

## 3. Building Blocks

Every agentic workflow implementation needs these primitives:

| Primitive | ReleaseFlow mapping |
|-----------|---------------------|
| **WorkflowState** | `goal`, `artifacts`, `messages`, `status` |
| **Step function** | `research_step(state) → state` |
| **Agent** | Researcher, Writer, Critic — callable specialists |
| **Router** | `if not approved: goto write` |
| **Termination** | `approved and human_ok → publish` |
| **Trace** | Append-only log of step names, timings, outcomes |

In [ ]:
@dataclass
class WorkflowMessage:
    sender: str
    content: str
    msg_type: str = "text"
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


@dataclass
class WorkflowState:
    """Shared state passed between workflow steps."""
    workflow_id: str
    goal: str
    artifacts: dict[str, Any] = field(default_factory=dict)
    messages: list[WorkflowMessage] = field(default_factory=list)
    status: str = "running"  # running | approved | rejected | published | failed
    retry_count: int = 0
    max_retries: int = 2

    def log(self, sender: str, content: str, msg_type: str = "text") -> None:
        self.messages.append(WorkflowMessage(sender=sender, content=content, msg_type=msg_type))

    def set_artifact(self, key: str, value: Any) -> None:
        self.artifacts[key] = value


state = WorkflowState(
    workflow_id=f"wf_{uuid.uuid4().hex[:8]}",
    goal="Produce v2.4.0 release announcement for customers",
)
print(f"Workflow {state.workflow_id}: {state.goal}")

---

## 4. Specialist Agents (Agentic Step Workers)

Each agent is a **specialist** with a narrow job. In production these wrap LLM calls; here they use deterministic logic so the notebook runs offline.

In [ ]:
def keyword_search(docs: list[dict[str, str]], query: str) -> list[dict[str, str]]:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    scored = []
    for doc in docs:
        text = " ".join(str(v) for v in doc.values()).lower()
        score = sum(1 for term in terms if term in text) if terms else 1
        scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    return [d for _, d in scored]


class ResearcherAgent:
    """Gathers facts from changelog and runbooks."""
    name = "researcher"

    def run(self, state: WorkflowState) -> WorkflowState:
        changes = [c["item"] for c in CHANGELOG]
        deploy = RUNBOOKS[0]["text"]
        comm_hits = keyword_search(RUNBOOKS, "customer communication")
        facts = {
            "version": "2.4.0",
            "changes": changes,
            "deploy_notes": deploy,
            "comm_standards": comm_hits[0]["text"] if comm_hits else "",
        }
        state.set_artifact("research", facts)
        state.log(self.name, f"Gathered {len(changes)} changelog items and deploy runbook.")
        return state


class WriterAgent:
    """Drafts customer announcement from research artifacts."""
    name = "writer"

    def run(self, state: WorkflowState) -> WorkflowState:
        facts = state.artifacts.get("research", {})
        changes = facts.get("changes", [])
        feedback = state.artifacts.get("review_feedback", "")
        draft = (
            f"Release v{facts.get('version', '?')} is now available.\n\n"
            f"What's new: " + "; ".join(changes) + ".\n\n"
            f"Breaking changes: Legacy v1 webhook format is deprecated.\n\n"
            f"Questions? Contact #releases."
        )
        if feedback and "too long" in feedback:
            draft = "\n".join(draft.split("\n")[:4]) + "\n\nQuestions? Contact #releases."
        state.set_artifact("draft", draft)
        state.log(self.name, f"Draft ready ({len(draft.split())} words)")
        return state


class CriticAgent:
    """Reviews draft against style guide; sets approved flag."""
    name = "critic"

    def run(self, state: WorkflowState) -> WorkflowState:
        draft = state.artifacts.get("draft", "")
        issues = []
        for required in STYLE_GUIDE["must_include"]:
            if required == "version number" and "v2.4" not in draft:
                issues.append("missing version number")
            if required == "breaking changes section" and "Breaking changes" not in draft:
                issues.append("missing breaking changes section")
            if required == "support channel" and "#releases" not in draft:
                issues.append("missing support channel")
        word_count = len(draft.split())
        if word_count > STYLE_GUIDE["max_words"]:
            issues.append(f"too long ({word_count} words)")

        if issues:
            state.set_artifact("approved", False)
            state.set_artifact("review_feedback", ", ".join(issues))
            state.log(self.name, f"REJECTED: {', '.join(issues)}", msg_type="review")
            state.status = "rejected"
        else:
            state.set_artifact("approved", True)
            state.set_artifact("review_feedback", "")
            state.log(self.name, "APPROVED: draft meets style guide", msg_type="review")
            state.status = "approved"
        return state


print("Agents: researcher, writer, critic")

---

## 5. A Minimal Workflow Engine

The **orchestrator** runs steps in order, logs traces, and handles errors. Steps are plain callables `(state) → state`.

In [ ]:
StepFn = Callable[[WorkflowState], WorkflowState]


@dataclass
class WorkflowStep:
    name: str
    fn: StepFn
    step_type: str = "agent"  # agent | deterministic | gate


@dataclass
class WorkflowTrace:
    step: int
    name: str
    step_type: str
    status: str
    ms: float = 0.0
    detail: str = ""


class WorkflowEngine:
    """Runs a list of steps sequentially with tracing."""

    def __init__(self, steps: list[WorkflowStep]) -> None:
        self.steps = steps
        self.trace: list[WorkflowTrace] = []

    def run(self, initial_state: WorkflowState) -> WorkflowState:
        state = initial_state
        self.trace = []
        for i, step in enumerate(self.steps, start=1):
            t0 = datetime.now(timezone.utc)
            try:
                state = step.fn(state)
                ms = (datetime.now(timezone.utc) - t0).total_seconds() * 1000
                self.trace.append(WorkflowTrace(
                    step=i, name=step.name, step_type=step.step_type,
                    status="ok", ms=round(ms, 2),
                ))
            except Exception as exc:
                ms = (datetime.now(timezone.utc) - t0).total_seconds() * 1000
                state.status = "failed"
                state.log("orchestrator", f"Step {step.name} failed: {exc}", msg_type="error")
                self.trace.append(WorkflowTrace(
                    step=i, name=step.name, step_type=step.step_type,
                    status="error", ms=round(ms, 2), detail=str(exc),
                ))
                break
        return state

    def trace_summary(self) -> list[str]:
        return [f"{t.step}. {t.name} ({t.step_type}) — {t.status} [{t.ms}ms]" for t in self.trace]


researcher = ResearcherAgent()
writer = WriterAgent()
critic = CriticAgent()

linear_steps = [
    WorkflowStep("research", researcher.run, "agent"),
    WorkflowStep("write", writer.run, "agent"),
    WorkflowStep("review", critic.run, "agent"),
]

engine = WorkflowEngine(linear_steps)
print(f"Engine ready with {len(linear_steps)} steps")

---

## 6. Linear Agentic Workflow — First Run

The simplest agentic workflow: **fixed order**, three agent steps, no branching.

In [ ]:
def run_linear_release_workflow() -> dict[str, Any]:
    state = WorkflowState(
        workflow_id=f"wf_{uuid.uuid4().hex[:8]}",
        goal="Produce v2.4.0 release announcement",
    )
    engine = WorkflowEngine(linear_steps)
    final = engine.run(state)
    return {
        "workflow_id": final.workflow_id,
        "status": final.status,
        "approved": final.artifacts.get("approved"),
        "draft_preview": final.artifacts.get("draft", "")[:200],
        "trace": engine.trace_summary(),
        "message_count": len(final.messages),
    }


result = run_linear_release_workflow()
print(f"Workflow: {result['workflow_id']}")
print(f"Status: {result['status']} | Approved: {result['approved']}")
print("Trace:")
for line in result["trace"]:
    print(f"  {line}")
print(f"\nDraft preview:\n{result['draft_preview']}...")

---

## 7. Branching — Review Loop Until Approved

Real agentic workflows **loop**: if the critic rejects, route back to the writer. The orchestrator owns this edge — not the model.

```
research ──► write ──► review ──► approved? ──yes──► publish
                      ▲              │
                      └──── no ──────┘  (retry_count < max)
```

In [ ]:
class ReleaseWorkflowOrchestrator:
    """Agentic workflow with review loop and human gate."""

    def __init__(
        self,
        *,
        auto_approve_human: bool = True,
        force_long_draft: bool = False,
    ) -> None:
        self.researcher = ResearcherAgent()
        self.writer = WriterAgent()
        self.critic = CriticAgent()
        self.auto_approve_human = auto_approve_human
        self.force_long_draft = force_long_draft
        self.trace: list[dict[str, Any]] = []

    def run(self, goal: str) -> dict[str, Any]:
        state = WorkflowState(workflow_id=f"wf_{uuid.uuid4().hex[:8]}", goal=goal)
        self.trace = []

        # Step 1: research (always)
        state = self._run_step("research", self.researcher.run, state)

        # Step 2–3: write ↔ review loop
        while state.retry_count <= state.max_retries:
            state = self._run_step("write", self.writer.run, state)
            if self.force_long_draft and state.retry_count == 0:
                state.artifacts["draft"] = state.artifacts.get("draft", "") + " " + "extra " * 80
            state = self._run_step("review", self.critic.run, state)
            if state.artifacts.get("approved"):
                break
            state.retry_count += 1
            self.trace.append({"step": "router", "action": "retry_write", "attempt": state.retry_count})

        # Step 4: human gate
        state = self._run_step("human_gate", self._human_gate, state)

        # Step 5: publish (deterministic)
        if state.status == "approved":
            state = self._run_step("publish", self._publish, state)

        return {
            "workflow_id": state.workflow_id,
            "status": state.status,
            "retries": state.retry_count,
            "draft": state.artifacts.get("draft", ""),
            "published": state.status == "published",
            "trace": self.trace,
        }

    def _run_step(self, name: str, fn: StepFn, state: WorkflowState) -> WorkflowState:
        state = fn(state)
        self.trace.append({"step": name, "status": state.status, "artifacts": list(state.artifacts.keys())})
        return state

    def _human_gate(self, state: WorkflowState) -> WorkflowState:
        if not state.artifacts.get("approved"):
            state.status = "failed"
            state.log("human_gate", "Skipped — draft not approved by critic")
            return state
        if self.auto_approve_human:
            state.log("human_gate", "Auto-approved for demo (production: await human click)")
        else:
            state.status = "failed"
            state.log("human_gate", "Awaiting human approval")
        return state

    def _publish(self, state: WorkflowState) -> WorkflowState:
        announcement = {
            "id": f"ANN-{uuid.uuid4().hex[:6].upper()}",
            "version": state.artifacts.get("research", {}).get("version"),
            "body": state.artifacts.get("draft", ""),
            "published_at": datetime.now(timezone.utc).isoformat(),
        }
        PUBLISHED_ANNOUNCEMENTS.append(announcement)
        state.status = "published"
        state.set_artifact("announcement_id", announcement["id"])
        state.log("publisher", f"Published {announcement['id']}")
        return state


orch = ReleaseWorkflowOrchestrator()
happy = orch.run("Produce v2.4.0 release announcement")
print(f"Status: {happy['status']} | Retries: {happy['retries']} | Published: {happy['published']}")
for t in happy["trace"]:
    print(f"  {t}")

In [ ]:
# Demo review loop — force an overlong draft so critic rejects once
retry_orch = ReleaseWorkflowOrchestrator(force_long_draft=True)
retry_result = retry_orch.run("Produce v2.4.0 release announcement with retry")
print(f"Status: {retry_result['status']} | Retries: {retry_result['retries']}")
router_events = [t for t in retry_result["trace"] if t.get("step") == "router"]
print(f"Router retries: {len(router_events)}")
print(f"Final word count: {len(retry_result['draft'].split())}")

---

## 8. Deterministic vs Agentic Steps

Not every node needs an LLM. Mix step types deliberately:

| Step type | Example | Who decides output? |
|-----------|---------|---------------------|
| **Deterministic** | `publish`, `validate_schema`, `fetch_from_db` | Code |
| **Agentic** | `research`, `write`, `review` | LLM (simulated here) |
| **Gate** | `human_gate`, `compliance_check` | Rule + optional human |
| **Router** | `if approved: publish else: retry` | Code |

**Rule:** Put guardrails and side effects in **deterministic** steps; put language generation in **agentic** steps.

In [ ]:
def classify_release_type(state: WorkflowState) -> WorkflowState:
    """Deterministic step — no LLM."""
    changes = state.artifacts.get("research", {}).get("changes", [])
    has_breaking = any("deprecat" in c.lower() or "breaking" in c.lower() for c in changes)
    state.set_artifact("release_type", "major" if has_breaking else "minor")
    state.log("classifier", f"Classified as {state.artifacts['release_type']} release")
    return state


def route_by_release_type(state: WorkflowState) -> str:
    """Router returns next step name."""
    return "extra_review" if state.artifacts.get("release_type") == "major" else "standard_review"


# Quick demo of deterministic classification
demo_state = WorkflowState(workflow_id="demo", goal="classify")
demo_state.set_artifact("research", {"changes": [c["item"] for c in CHANGELOG]})
classify_release_type(demo_state)
print(f"Release type: {demo_state.artifacts['release_type']}")
print(f"Next route: {route_by_release_type(demo_state)}")

---

## 9. State Passing and Hand-off Contracts

Steps communicate through **shared state**, not ad-hoc globals. Define **contracts** — what each step reads and writes:

| Step | Reads | Writes |
|------|-------|--------|
| `research` | `goal` | `artifacts.research` |
| `write` | `artifacts.research`, `artifacts.review_feedback` | `artifacts.draft` |
| `review` | `artifacts.draft`, `STYLE_GUIDE` | `artifacts.approved`, `artifacts.review_feedback` |
| `publish` | `artifacts.draft`, `artifacts.approved` | external `PUBLISHED_ANNOUNCEMENTS` |

Violating contracts (e.g. `write` before `research`) causes empty drafts — the orchestrator must enforce valid order.

In [ ]:
STEP_CONTRACTS: dict[str, dict[str, list[str]]] = {
    "research": {"reads": ["goal"], "writes": ["research"]},
    "write": {"reads": ["research", "review_feedback"], "writes": ["draft"]},
    "review": {"reads": ["draft"], "writes": ["approved", "review_feedback"]},
    "publish": {"reads": ["draft", "approved"], "writes": ["announcement_id"]},
}


def validate_handoff(step_name: str, state: WorkflowState) -> list[str]:
    """Check required reads exist before a step runs."""
    contract = STEP_CONTRACTS.get(step_name, {})
    missing = [key for key in contract.get("reads", []) if key != "goal" and key not in state.artifacts]
    return missing


empty_state = WorkflowState(workflow_id="test", goal="test")
print("write before research:", validate_handoff("write", empty_state))

filled = WorkflowState(workflow_id="test", goal="test")
filled.set_artifact("research", {"version": "2.4.0"})
print("write after research:", validate_handoff("write", filled))

---

## 10. Common Workflow Topologies (Preview)

Agentic workflows compose a few recurring shapes. Later material in this track goes deeper on each; here is the map:

```
(a) Sequential          (b) Review loop         (c) Supervisor

 A ──► B ──► C          A ──► B ──► C ──► B     Manager ──► Worker A
                                              └──► Worker B ──► Manager

(d) Parallel fan-out    (e) Nested workflow

       ┌──► Legal                           Parent workflow
 A ────┼──► Security      ──► Merge              │
       └──► Technical                     sub-workflow (research only)
```

| Topology | Orchestrator decides | Example |
|----------|---------------------|--------|
| **Sequential** | Fixed A→B→C | ReleaseFlow research→write→review |
| **Loop** | Retry until condition | Critic reject → rewrite |
| **Supervisor** | Which worker next | Manager routes to researcher or writer |
| **Parallel** | Run branches, then merge | Legal + security review concurrently |
| **Nested** | Sub-workflow as one step | Research sub-pipeline inside larger flow |

---

## 11. Termination Conditions

Every agentic workflow needs explicit **stop rules**:

| Condition | ReleaseFlow example |
|-----------|---------------------|
| **Success** | `status == published` |
| **Max retries** | `retry_count > max_retries` → fail gracefully |
| **Human rejection** | Human gate returns false |
| **Budget / timeout** | Total workflow wall-clock exceeded |
| **Unrecoverable error** | Step raises exception |

Without termination rules, review loops can run forever.

In [ ]:
@dataclass
class TerminationPolicy:
    max_retries: int = 2
    max_wall_seconds: float = 300.0
    require_human_approval: bool = True

    def should_stop_retry(self, state: WorkflowState) -> bool:
        return state.retry_count > self.max_retries

    def should_publish(self, state: WorkflowState, human_ok: bool) -> bool:
        return (
            state.artifacts.get("approved") is True
            and (human_ok or not self.require_human_approval)
        )


policy = TerminationPolicy(max_retries=2)
test_state = WorkflowState(workflow_id="t", goal="t", retry_count=3)
print(f"Stop retry at 3: {policy.should_stop_retry(test_state)}")
test_state.retry_count = 1
test_state.set_artifact("approved", True)
print(f"Can publish: {policy.should_publish(test_state, human_ok=True)}")

---

## 12. Observability — Workflow Traces

Production workflows log **structured traces** — not just final output:

```json
{"step": 1, "name": "research", "type": "agent", "status": "ok", "ms": 12}
{"step": 2, "name": "write", "type": "agent", "status": "ok", "ms": 8}
{"step": "router", "action": "retry_write", "attempt": 1}
{"step": 5, "name": "publish", "type": "deterministic", "status": "ok"}
```

Traces power debugging ("why did we retry?"), SLAs ("p95 write step latency"), and compliance audits.

In [ ]:
def trace_to_json_lines(trace: list[dict[str, Any]]) -> str:
    return "\n".join(json.dumps(e, default=str) for e in trace)


def workflow_dag_ascii(trace: list[dict[str, Any]]) -> str:
    names = []
    for t in trace:
        s = t.get("step")
        if s == "router":
            names.append(f"↺ retry({t.get('attempt')})")
        elif isinstance(s, str):
            names.append(s)
    return " → ".join(names)


retry_orch2 = ReleaseWorkflowOrchestrator(force_long_draft=True)
r = retry_orch2.run("trace demo")
print("DAG:", workflow_dag_ascii(r["trace"]))
print("\nJSON lines (first 3):")
print("\n".join(trace_to_json_lines(r["trace"]).split("\n")[:3]))

---

## 13. Agentic Workflow vs Autonomous Agent — Same Task

Compare solving "produce a release announcement" two ways:

In [ ]:
class AutonomousReleaseAgent:
    """Model (simulated) picks next action freely — no fixed graph."""

    def __init__(self) -> None:
        self.researcher = ResearcherAgent()
        self.writer = WriterAgent()
        self.critic = CriticAgent()
        self.trace: list[str] = []

    def run(self, goal: str) -> dict[str, Any]:
        state = WorkflowState(workflow_id="agent", goal=goal)
        # Planner decides order — could vary between runs with a real LLM
        plan = ["research", "write", "review", "write", "review"]  # simulated re-plan
        agents = {"research": self.researcher, "write": self.writer, "review": self.critic}
        for action in plan:
            self.trace.append(action)
            state = agents[action].run(state)
            if action == "review" and state.artifacts.get("approved"):
                break
        return {"mode": "autonomous", "trace": self.trace, "approved": state.artifacts.get("approved")}


workflow_run = ReleaseWorkflowOrchestrator().run("compare")
agent_run = AutonomousReleaseAgent().run("compare")

COMPARISON = [
    ("Control", "Orchestrator graph", "Planner picks next"),
    ("Trace shape", "Named steps + router", "Action sequence"),
    ("Human gate", "Explicit step", "Must add guardrail"),
    ("Retries", "retry_count in state", "Implicit in plan"),
    ("Approved", str(workflow_run["status"]), str(agent_run["approved"])),
]

print(f"{'Dimension':<14} {'Agentic workflow':<22} {'Autonomous agent'}")
print("-" * 58)
for dim, wf, ag in COMPARISON:
    print(f"{dim:<14} {wf:<22} {ag}")

---

## 14. When to Use Agentic Workflows

| Signal | Use agentic workflow |
|--------|----------------------|
| Process has **known stages** with occasional rework | ✅ Review loops, approval gates |
| **Compliance** requires auditable step order | ✅ Trace each node |
| Different **specialists** with different prompts | ✅ Researcher vs writer vs critic |
| Task path is **unknown upfront** | ❌ Prefer autonomous agent or hybrid |
| **Zero LLM** needed | ❌ Pure deterministic workflow is simpler |
| Single generalist can do it in one prompt | ❌ Single agent may suffice |

**Hybrid** is common: workflow shell (classify → gate) + short agent loop in the middle for messy subtasks.

---

## 15. Early Anti-Patterns to Avoid

| Anti-pattern | Symptom | Fix |
|--------------|---------|-----|
| **Agent everywhere** | Slow, expensive, hard to test | Use deterministic steps where possible |
| **No shared state contract** | Steps read missing artifacts | Document reads/writes per step |
| **Unbounded loops** | Runaway retries | `max_retries` + termination policy |
| **Implicit routing** | Model decides graph edges | Orchestrator owns branches |
| **No trace** | "It failed somewhere" | Structured log per step |
| **Skipping human gates** | Publishes bad content | Gate before side effects |

---

## 16. Optional Live LLM for Agentic Steps

Set `USE_LIVE_LLM = True` to replace the offline `WriterAgent` with an OpenAI call. The **orchestrator graph stays identical** — only the agent step implementation changes.

In [ ]:
USE_LIVE_LLM = False


class LiveWriterAgent:
    """Writer step backed by OpenAI — drop-in replacement for WriterAgent."""
    name = "writer"

    def run(self, state: WorkflowState) -> WorkflowState:
        facts = state.artifacts.get("research", {})
        feedback = state.artifacts.get("review_feedback", "")
        prompt = (
            f"Write a customer release announcement for v{facts.get('version')}.\n"
            f"Changes: {facts.get('changes')}\n"
            f"Standards: {facts.get('comm_standards', '')}\n"
            f"Feedback to address: {feedback or 'none'}\n"
            f"Include breaking changes section and #releases support channel. Max 200 words."
        )
        try:
            from openai import OpenAI
            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
            )
            draft = resp.choices[0].message.content or ""
        except Exception as exc:
            draft = f"[LLM error: {exc}]"
        state.set_artifact("draft", draft)
        state.log(self.name, f"Live draft ({len(draft.split())} words)")
        return state


if USE_LIVE_LLM:
    live_orch = ReleaseWorkflowOrchestrator()
    live_orch.writer = LiveWriterAgent()
    live_result = live_orch.run("Live LLM release announcement")
    print(live_result["draft"][:400])
else:
    print("Offline mode — set USE_LIVE_LLM = True to use OpenAI for the writer step.")
    print(f"Published announcements so far: {len(PUBLISHED_ANNOUNCEMENTS)}")

---

## 17. Quiz

1. What two ideas does an agentic workflow combine?
2. Who owns the graph edges — the orchestrator or the model?
3. What is a **hand-off contract** between steps?
4. Name three termination conditions for a workflow.
5. When would you choose an agentic workflow over a fully autonomous agent?

---

## 18. Summary

| Concept | Takeaway |
|---------|----------|
| Agentic workflow | Orchestrated graph + LLM steps at designated nodes |
| vs pure workflow | Adds non-deterministic intelligence where language matters |
| vs autonomous agent | You keep control flow; model works inside steps |
| Building blocks | State, steps, agents, router, termination, trace |
| ReleaseFlow | research → write ↔ review → human_gate → publish |
| Contracts | Each step declares what it reads and writes |
| Topologies | Sequential, loop, supervisor, parallel, nested |

Reuse `WorkflowState`, `WorkflowEngine`, and `ReleaseWorkflowOrchestrator` as templates: swap agent implementations for live LLMs, add parallel branches, or insert supervisor routing as your processes grow.